# KnowWow - Work as usual, knowledge grows

**제조 업무의 반복 패턴과 다른 처리가 발생했을 때만 짧게 질문해 판단 조건을 지식으로 남기는 AI 도우미**

- 핵심 기능: `업무 기록 입력 → 비슷한 과거 처리 집계 → 차이가 있을 때 AI 질문 → 담당자 답변 → 조건 후보 추출 → 사람 확인 후 저장`
- 데이터: 의도적으로 설계한 가상 제조 업무 Comment 24건. 실제 기업 데이터가 아닙니다.
- 범위: 온톨로지 구축은 제외하고, 하루 MVP에 맞춰 JSON 데이터와 5개 비교 항목을 사용
- 안전 원칙: LLM은 패턴이나 정답을 만들지 않고 **질문 생성·답변 구조화·검색 결과 설명**만 담당

> 루트 `.env`의 실제 API 키로 모든 코드를 실행하고, 질문 생성·답변 구조화 결과를 셀 출력에 저장했습니다. 업무 기록과 담당자 답변은 가상 데이터이며, 표시된 AI 질문·추출 결과는 실제 모델 호출 결과입니다. 재실행은 저장소의 `README.md`를 따라 의존성을 설치하고 루트 `.env`에 본인 키를 넣은 뒤, 프로젝트 루트나 `제출파일` 폴더에서 노트북을 열어 순서대로 실행하면 됩니다. 키 없이도 저장된 출력은 읽을 수 있습니다.


## 1. 기획 의도 — 왜 LLM인가?

**한 문장으로:** 제조 업무 담당자가 과거와 다른 방식으로 처리한 이유를 따로 기록하지 않아 경험이 사라지는 문제를, 필요한 순간에만 짧게 묻고 정리하는 도우미입니다.

예를 들어 세면대 온수 배관이 빠졌고 자재와 도면이 준비된 업무들은 주로 생산 부서로 넘겼습니다. 그런데 `CASE-008`은 도면을 고쳤습니다. 기록에는 *무엇을 했는지*만 있고 *왜 달랐는지*는 없습니다. 단순 검색과 규칙은 건수와 차이까지 찾을 수 있지만, 기록마다 다른 문장을 읽고 담당자에게 자연스럽고 답을 유도하지 않는 질문을 만들거나 자유로운 답변에서 새 조건을 추출하기 어렵습니다. 그래서 **차이 감지는 코드**, **질문과 답변 해석은 LLM**에 맡겼습니다.

| 비교에 쓰는 데이터 | 뜻 | 예시 (`CASE-008`) |
|---|---|---|
| 문제 유형·장비·계통 | 어떤 문제, 어느 설비인가 | 설치 누락·세면대·온수 계통 |
| 자재 상태·도면 상태 | 당시 이미 알려진 조건 | 자재 있음·도면 유효 |
| 처리 방식 | 실제로 택한 조치 | 도면 개정 |
| 원문·답변 기록 | 사람이 남긴 설명 | 누락 Comment와 처리 Response |

앞의 5개 항목은 비슷한 상황을 **묶는 기준**이고, 처리 방식은 그 안에서 **결과를 비교하는 값**입니다. 원문은 질문을 만들 때 참고합니다. 이 데이터는 실제 업무 규칙이나 정답이 아니라 MVP를 확인하기 위한 예시입니다.


In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (candidate / "data/comment_cases.json").is_file()
                     and (candidate / "ai-service").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("KnowWow 프로젝트 폴더 또는 제출파일 폴더에서 실행해주세요.")

sys.path.insert(0, str(PROJECT_ROOT / "ai-service"))
cases = json.loads((PROJECT_ROOT / "data/comment_cases.json").read_text(encoding="utf-8"))
scenarios = json.loads((PROJECT_ROOT / "data/demo_scenarios.json").read_text(encoding="utf-8"))

print("project: KnowWow")
print(f"synthetic cases: {len(cases)}")
print(f"demo scenarios: {len(scenarios)}")


project: KnowWow
synthetic cases: 24
demo scenarios: 3


## 2. Deterministic Context Engineering

LLM에 원시 데이터 전체를 넘기지 않습니다. 먼저 코드가 아래 5개 필드로 같은 상황을 묶고, Action 분포를 계산합니다.

`issue_type + equipment + system + material_status + drawing_status`

지원 사례 3건 이상, 가장 많은 처리의 비율 67% 이상일 때만 비교할 만한 반복 경향으로 취급합니다. 현재 처리가 그 방식과 다를 때 `ACTION_VARIANT`(과거 다수와 다른 처리)로 판정하며, 이 판정에는 LLM을 사용하지 않습니다. 아래 결과에서 `CASE-001`은 다수 방식 그대로 처리해 질문하지 않고, `CASE-008`은 도면 개정으로 달라 질문합니다. 이 비율은 **관찰값**이지 현장 표준이나 정답이 아닙니다.


In [2]:
SIGNATURE_FIELDS = ("issue_type", "equipment", "system")

def signature(item):
    return (
        *(item[field] for field in SIGNATURE_FIELDS),
        item["context"]["material_status"],
        item["context"]["drawing_status"],
    )

grouped = defaultdict(list)
for item in cases:
    grouped[signature(item)].append(item)

patterns = []
for index, (_, members) in enumerate(
    sorted(grouped.items(), key=lambda pair: min(item["case_id"] for item in pair[1])), start=1
):
    if len(members) < 2:
        continue
    distribution = Counter(item["action"] for item in members)
    majority_action, majority_count = sorted(
        distribution.items(), key=lambda pair: (-pair[1], pair[0])
    )[0]
    ratio = majority_count / len(members)
    patterns.append({
        "pattern_id": f"PATTERN-{index:03d}",
        "signature": {
            "issue_type": members[0]["issue_type"],
            "equipment": members[0]["equipment"],
            "system": members[0]["system"],
            "material_status": members[0]["context"]["material_status"],
            "drawing_status": members[0]["context"]["drawing_status"],
        },
        "support_count": len(members),
        "action_distribution": dict(distribution),
        "majority_action": majority_action,
        "majority_ratio": ratio,
        "supporting_case_ids": [item["case_id"] for item in members],
        "stable": len(members) >= 3 and ratio >= 0.67,
    })

print(f"mined patterns: {len(patterns)}")
for pattern in patterns:
    print(pattern["pattern_id"], pattern["support_count"], pattern["action_distribution"], pattern["stable"])


mined patterns: 4
PATTERN-001 8 {'TRANSFER_TO_PRODUCTION': 6, 'SITE_CHECK_THEN_TRANSFER': 1, 'DRAWING_REVISION': 1} True
PATTERN-002 5 {'DRAWING_REVISION': 4, 'REQUEST_CLARIFICATION': 1} True
PATTERN-003 5 {'MATERIAL_REQUEST': 4, 'TRANSFER_TO_PRODUCTION': 1} True
PATTERN-004 6 {'REQUEST_CLARIFICATION': 5, 'DRAWING_REVISION': 1} True


In [3]:
pattern_by_signature = {tuple(pattern["signature"].values()): pattern for pattern in patterns}

def detect_gap(case):
    pattern = pattern_by_signature.get(signature(case))
    if pattern is None:
        return {"status": "NO_PATTERN", "requires_interview": False}
    variant = pattern["stable"] and case["action"] != pattern["majority_action"]
    return {
        "status": "ACTION_VARIANT" if variant else "NONE",
        "requires_interview": variant,
        "current_action": case["action"],
        "majority_action": pattern["majority_action"],
        "pattern_id": pattern["pattern_id"],
    }

for case_id in ("CASE-001", "CASE-008"):
    item = next(item for item in cases if item["case_id"] == case_id)
    print(case_id, detect_gap(item))


CASE-001 {'status': 'NONE', 'requires_interview': False, 'current_action': 'TRANSFER_TO_PRODUCTION', 'majority_action': 'TRANSFER_TO_PRODUCTION', 'pattern_id': 'PATTERN-001'}
CASE-008 {'status': 'ACTION_VARIANT', 'requires_interview': True, 'current_action': 'DRAWING_REVISION', 'majority_action': 'TRANSFER_TO_PRODUCTION', 'pattern_id': 'PATTERN-001'}


## 3. 핵심 LangChain 체인: 입력 → 프롬프트 → LLM → 질문

`ChatPromptTemplate`의 system 메시지는 AI를 '제조 현장 경험 기록을 돕는 사람'으로 설정하고, **이유 추측 금지·답 유도 금지·쉬운 한국어·한 문장 질문**을 요구합니다. human 메시지에는 이번 기록과 과거 처리 건수만 넣습니다. 모델이 이유를 먼저 지어내면 담당자의 답변도 그쪽으로 유도될 수 있기 때문입니다. 구체적 원인을 담은 예시 질문은 의도적으로 넣지 않았습니다. `StrOutputParser`는 모델의 채팅 메시지에서 질문 문자열을 꺼내 화면에 전달합니다. 프롬프트와 체인을 분리하지 않으면 이 제약과 입력 형식을 여러 사례에 일관되게 적용하기 어렵습니다.

이 제출 노트북은 실제 API 키가 있어야 실행됩니다. 가짜 질문을 출력하는 대체 모델은 사용하지 않습니다. `ChatPromptTemplate | ChatModel | StrOutputParser`를 실제 모델로 실행한 결과를 아래에 저장했습니다.


In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from app.knowledge_service import micro_question_context
from app.models import MicroQuestionRequest
from app.terminology import ACTION_LABELS, humanize_chat_text, label_of

load_dotenv(PROJECT_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY", "").strip():
    raise RuntimeError("실제 질문 생성 결과를 저장하려면 루트 .env에 OPENAI_API_KEY를 입력해주세요.")

micro_question_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 제조 현장의 경험을 기록하도록 돕는 AI입니다.\n"
        "이번 업무의 처리 이유를 추측하지 마십시오.\n"
        "비슷한 과거 업무의 처리와 이번 처리의 차이를 확인하는 질문 한 개만 만드십시오.\n"
        "원인 예시를 제시하거나 답을 유도하지 마십시오.\n"
        "입력에 제공된 쉬운 한국어 표현을 그대로 사용하십시오.\n"
        "영문 코드나 Case, Action, Context, Pattern 같은 시스템 용어는 쓰지 마십시오.\n"
        "과거에 가장 많았던 처리와 이번 처리를 언급하고 어떤 상황이 달랐는지 물으십시오.\n"
        "'도면 개정'은 정확히 '도면 개정'이라고 쓰고, 질문 한 문장만 출력하십시오."
    )),
    ("human", "이번 업무:\n{current_case}\n\n비슷한 과거 업무에서 관찰된 내용:\n{matched_pattern}"),
])

model = init_chat_model(
    model=os.getenv("MODEL_NAME", "gpt-4o-mini"),
    model_provider=os.getenv("MODEL_PROVIDER", "openai"),
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

micro_question_chain = micro_question_prompt | model | StrOutputParser()
print("LIVE_MODE - 실제 모델 호출")
print("chain: ChatPromptTemplate | ChatModel | StrOutputParser")


LIVE_MODE - 실제 모델 호출
chain: ChatPromptTemplate | ChatModel | StrOutputParser


In [5]:
target_case = next(item for item in cases if item["case_id"] == "CASE-008")
target_pattern = pattern_by_signature[signature(target_case)]
request = MicroQuestionRequest.model_validate({
    "current_case": target_case,
    "matched_pattern": target_pattern,
})
current_case_prompt, matched_pattern_prompt = micro_question_context(request)

question = humanize_chat_text(micro_question_chain.invoke({
    "current_case": json.dumps(current_case_prompt, ensure_ascii=False, indent=2),
    "matched_pattern": json.dumps(matched_pattern_prompt, ensure_ascii=False, indent=2),
}).strip())

print("입력 Case:", target_case["case_id"])
print("Gap:", detect_gap(target_case)["status"])
print("LLM 질문:", question)


입력 Case: CASE-008
Gap: ACTION_VARIANT
LLM 질문: 가장 많이 했던 처리인 '생산 부서로 넘겨 처리'와 이번 처리인 '도면 개정'의 상황은 어떤 점에서 달랐나요?


### 대표 입력 `CASE-008`: 질문이 나오기까지

| 단계 | 이번 실행에서 사용한 내용 |
|---|---|
| 입력 | 세면대 온수 배관 설치 누락. 자재는 있고 도면은 유효한데, 이번에는 **도면 개정**으로 처리 |
| 컨텍스트 구성 | 코드가 같은 5개 비교 항목의 과거 8건을 묶음: 생산 부서로 넘김 6건, 현장 확인 후 넘김 1건, 도면 개정 1건. 이번 처리는 다수와 달라 질문 대상으로 판정 |
| 체인 실행 | `micro_question_context()`가 원문·현재 처리·알려진 자재/도면 상황·과거 건수를 쉬운 한국어 입력으로 만들고, `ChatPromptTemplate → ChatModel → StrOutputParser`에 전달 |
| 후처리 | `strip()`으로 공백을 정리하고 `humanize_chat_text()`로 혹시 남은 내부 코드 표현을 사용자용 용어로 바꿈 |
| 출력 | 위 셀에 저장된 실제 질문: 가장 많았던 처리와 이번 도면 개정 사이에 어떤 상황 차이가 있었는지 확인 |

이 단계에서 AI가 **도면을 고쳐야 했던 이유**를 결정한 것은 아닙니다. 그 이유는 다음 셀의 담당자 답변으로만 확인합니다.


## 4. Structured Output — 답변을 저장 가능한 지식으로

사용자 답변을 바로 지식으로 넣지 않고 Pydantic 스키마(`new_context`, `rationale`, `exception`)로 제한합니다. `model.with_structured_output(PersonalKnowledgeExtraction)`으로 실제 모델 결과를 받고, 답변에 없는 값은 추측하지 말고 `null`로 두도록 별도 프롬프트에 지시합니다. 자유 형식 텍스트만 받으면 조건명과 근거를 저장·검색하기 어렵습니다. 아래 셀은 **저장 전 미리보기**까지 실행하며, 화면에서 사람이 확인해야 개인 지식으로 저장됩니다.


In [6]:
from app.models import PersonalKnowledgeExtraction
from app.prompts import KNOWLEDGE_EXTRACTION_PROMPT

user_answer = "실제 설치 위치에 다른 장비가 있어서 그대로 설치할 수 없었습니다."

extraction_chain = KNOWLEDGE_EXTRACTION_PROMPT | model.with_structured_output(PersonalKnowledgeExtraction)
extracted = extraction_chain.invoke({
    "current_case": json.dumps(target_case, ensure_ascii=False, indent=2),
    "matched_pattern": json.dumps(target_pattern, ensure_ascii=False, indent=2),
    "question": question,
    "answer": user_answer,
})

print(extracted.model_dump_json(indent=2))
print("저장 정책: 사용자 확인 전에는 Personal Knowledge로 확정하지 않음")


{
  "new_context": {
    "name": "installation_location_issue",
    "value": "EQUIPMENT_CONFLICT"
  },
  "rationale": "설치 위치에 다른 장비가 있어 설치가 불가능했다는 점에서 두 처리 방식이 달랐다.",
  "exception": null
}
저장 정책: 사용자 확인 전에는 Personal Knowledge로 확정하지 않음


담당자가 '설치 위치에 다른 장비가 있었다'고 답하자, 모델은 **장비 간섭으로 설치가 어려웠다**는 조건 후보와 한국어 근거를 반환했습니다. 출력의 `installation_location_issue=EQUIPMENT_CONFLICT`는 내부 저장용 이름이며 화면에서는 쉬운 한국어로 설명합니다. 평가용 기대값(`installation_feasibility=IMPOSSIBLE`)과 이름·값은 다르므로 정확 일치로 채점하면 오답입니다. 의미상 핵심은 포착했지만, **표준화와 사람 확인이 필요**하다는 결과입니다. 이 노트북에서 실제 저장 버튼이나 DB 쓰기는 실행하지 않았습니다.


## 5. Retriever · Vector Store · Tool-calling Agent

질문·답변 구조화 외에, 축적된 경험을 나중에 찾아볼 때 근거를 잃지 않도록 다음 컴포넌트를 사용했습니다.

| 컴포넌트 | 배치한 이유 / 없으면 생기는 문제 | 이 노트북에서 확인하는 범위 |
|---|---|---|
| `Document`와 metadata | 사례 본문과 출처 ID를 같이 보관해 검색 결과의 근거를 추적 | 24건 문서 생성 |
| `OpenAIEmbeddings` + `InMemoryVectorStore` | 원문 표현이 달라도 의미가 비슷한 사례를 찾음. 단순 키워드 일치만으로는 놓칠 수 있음 | 실제 임베딩 API로 24건 색인 |
| `@tool` | 검색 함수를 AI가 호출할 수 있는 명시적 도구로 포장. 없으면 에이전트가 검색을 선택할 수 없음 | 유사 사례 검색 도구 등록 |
| `create_agent`와 검색 후 검증 | 앱에서는 사례·확인된 개인 지식·조직 경향 검색을 나눠 호출하고, 반환한 출처 ID가 실제 검색된 ID인지 서버에서 확인 | **앱 코드의 구현 범위**. 이 노트북은 에이전트 호출 결과를 시연하지 않음 |

아래 셀은 실제 임베딩 API로 24개 예시 업무 문서를 등록합니다. 데이터는 가상이지만 임베딩 호출과 색인은 실제입니다. 이 셀의 출력은 **색인 완료·도구 등록**만 증명하며 검색 품질이나 에이전트 답변을 검증한 결과는 아닙니다.


In [7]:
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = [
    Document(
        page_content=(
            f"과거 업무 사례 {item['case_id']}. 이슈 {item['issue_type']}. "
            f"Comment: {item['comment_text']} Action: {item['action']} Outcome: {item['outcome']}."
        ),
        metadata={"doc_type": "CASE", "source_id": item["case_id"]},
    )
    for item in cases
]

embeddings = OpenAIEmbeddings(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    api_key=os.getenv("OPENAI_API_KEY"),
)
case_store = InMemoryVectorStore(embedding=embeddings)
case_store.add_documents(documents)

@tool
def search_similar_cases(query: str, k: int = 5) -> str:
    '''의미가 비슷한 과거 제조 업무 Case와 실제 Source ID를 검색합니다.'''
    found = case_store.similarity_search(query, k=min(k, 5))
    return json.dumps([
        {"source_id": doc.metadata["source_id"], "content": doc.page_content}
        for doc in found
    ], ensure_ascii=False)

print("indexed documents:", len(documents))
print("registered tool:", search_similar_cases.name)
print("production tools: search_similar_cases, search_personal_knowledge, search_org_patterns, get_case_detail")


indexed documents: 24
registered tool: search_similar_cases
production tools: search_similar_cases, search_personal_knowledge, search_org_patterns, get_case_detail


## 6. 입력을 바꾼 테스트 시나리오 비교

대표 사례 외에 처리 방식과 사용자 답변이 다른 2건을 같은 질문·구조화 체인으로 다시 실행합니다. 아래 기대 조건은 평가용 데이터일 뿐 모델에 입력하지 않습니다. 실제 출력과 비교해 잘된 부분과 오차를 확인합니다.


In [8]:
for scenario in scenarios:
    if scenario["case_id"] == target_case["case_id"]:
        continue
    item = next(case for case in cases if case["case_id"] == scenario["case_id"])
    matched = pattern_by_signature[signature(item)]
    assert detect_gap(item)["requires_interview"]
    request = MicroQuestionRequest.model_validate({"current_case": item, "matched_pattern": matched})
    current_prompt, pattern_prompt = micro_question_context(request)
    generated = humanize_chat_text(micro_question_chain.invoke({
        "current_case": json.dumps(current_prompt, ensure_ascii=False, indent=2),
        "matched_pattern": json.dumps(pattern_prompt, ensure_ascii=False, indent=2),
    }).strip())
    structured = extraction_chain.invoke({
        "current_case": json.dumps(item, ensure_ascii=False, indent=2),
        "matched_pattern": json.dumps(matched, ensure_ascii=False, indent=2),
        "question": generated,
        "answer": scenario["answer"],
    })
    print(f"\n[{item['case_id']}]")
    print("이번 처리:", label_of(ACTION_LABELS, item["action"], "기타 처리"))
    print("가장 많았던 처리:", label_of(ACTION_LABELS, matched["majority_action"], "기타 처리"))
    print("AI 질문:", generated)
    print("담당자 답변:", scenario["answer"])
    print("AI가 정리한 근거:", structured.rationale)
    print("추출한 조건:", structured.new_context.model_dump() if structured.new_context else None)
    print("평가용 기대 조건:", scenario["expected_context_name"], scenario["expected_context_value"])



[CASE-018]
이번 처리: 생산 부서로 넘겨 처리
가장 많았던 처리: 필요한 자재 요청
AI 질문: 가장 많이 했던 처리인 '필요한 자재 요청'과 이번에 한 '생산 부서로 넘겨 처리'는 어떤 상황이 달랐나요?
담당자 답변: 동일 사양의 대체 자재가 이미 현장에 확보되어 있었습니다.
AI가 정리한 근거: 대체 자재가 확보되어 있어 처리 방식이 달라졌다.
추출한 조건: {'name': 'alternative_material_available', 'value': 'YES'}
평가용 기대 조건: substitute_material_availability AVAILABLE

[CASE-024]
이번 처리: 도면 개정
가장 많았던 처리: 관련 내용 추가 확인
AI 질문: 가장 많이 했던 처리인 '관련 내용 추가 확인'과 이번 처리인 '도면 개정'의 상황은 어떻게 달랐나요?
담당자 답변: 기본설계 단계에서 선주와 합의한 변경사항을 최종 합의 메일에서 확인했습니다.
AI가 정리한 근거: 기본설계 단계에서 선주와 합의한 변경사항을 확인했다는 점이 이번 처리와 관련된 상황이다.
추출한 조건: None
평가용 기대 조건: owner_approved_change CONFIRMED


### 이번 실행 결과에서 확인한 점

| 바꾼 입력 | 과거에 많았던 처리 → 이번 처리 | 담당자 답변의 핵심 | 실제 추출 결과 | 기대 조건과 비교 |
|---|---|---|---|---|
| `CASE-008` 설치 누락 | 생산 부서로 넘김 → 도면 개정 | 설치 위치에 다른 장비가 있음 | `installation_location_issue=EQUIPMENT_CONFLICT` | 장비 간섭은 포착. 기대한 `installation_feasibility=IMPOSSIBLE`과 **이름·값 불일치** |
| `CASE-018` 자재 문제 | 자재 요청 → 생산 부서로 넘김 | 대체 자재 확보 | `alternative_material_available=YES` | 의미는 포착. 기대한 `substitute_material_availability=AVAILABLE`과 **이름·값 불일치** |
| `CASE-024` 사양 충돌 | 추가 확인 → 도면 개정 | 선주 합의 메일 확인 | 근거 문장에 반영, 새 조건 `null` | 기대한 `owner_approved_change=CONFIRMED`를 **추출하지 못함** |

세 입력을 같은 체인으로 실행했지만 질문과 추출 결과가 달라졌습니다. 질문은 모두 과거 다수 처리와 이번 처리의 차이를 묻는 형태였고, 새 조건의 **의미 추출은 2건**, 기대 조건명·값의 **정확 일치는 0건**, **누락은 1건**입니다. 이는 저장된 이번 실행에 한정한 관찰이지 일반 성능 지표는 아닙니다. 평가용 기대 조건은 프롬프트에 전달하지 않았고, 실제 모델 응답은 재실행 시 달라질 수 있습니다.


## 7. 구현 범위와 한계

**구현된 범위**

1. 이 **노트북**: 24건 가상 데이터 로딩, 반복 경향·차이 판정 재현, 실제 LLM 질문·답변 구조화 3건, 실제 임베딩 색인과 검색 도구 등록
2. **웹 앱**: Spring Boot의 같은 취지의 판정 API, LangChain 기반 AI 서비스, Next.js 업무·질문·지식·검색 화면, 사용자 확인 후 개인 지식 JSON 저장
3. **미구현**: 개인 지식의 팀 지식 자동 승격, 온톨로지·지식 그래프, 운영 DB

**현재 한계**

- 온톨로지와 지식 그래프는 하루 MVP 범위에서 제외했습니다.
- 패턴 임계값(지원 3건, 67%)은 작은 Synthetic Data에 맞춘 값이라 운영 데이터로 재검증해야 합니다.
- In-memory Vector Store는 재시작하면 재구축되며 대규모 데이터에 적합하지 않습니다.
- 실제 테스트에서 두 건은 새 조건의 의미를 잡았어도 조건명·값이 기대 표기와 달랐고, 한 건은 새 조건이 `null`이었습니다. 자유로운 이름 생성 뒤에 **표준 용어 매핑·검증**이 필요합니다.
- `CASE-024`처럼 답변에 단서가 있는데도 조건 필드가 비면, 바로 저장하지 않고 담당자에게 재확인하거나 미완료 후보로 표시해야 합니다.
- 본 노트북은 실제 모델로 3건을 실행했지만, 작은 예시만으로 질문·추출 품질을 일반화할 수 없습니다. 검색 에이전트의 답변 품질도 여기서는 평가하지 않았습니다.
- 개인이 확인한 경험은 현재 개인 지식으로 저장됩니다. 여러 사례·담당자의 근거와 반례를 비교해 **팀 패턴으로 검토·승인하는 과정은 향후 과제**이며, 자동으로 팀의 정답이 되지 않습니다.

**다음 개선**: ① 조건명·값의 표준 목록과 매핑/재질문 ② 다양한 실제 업무를 포함한 평가 데이터셋 ③ 개인 지식→팀 패턴 검토·승인 워크플로 ④ 운영 DB·영속 Vector DB·실행 추적. 충분히 검증된 조건이 쌓인 뒤에야 온톨로지 후보를 검토합니다.
